# 7. Atenciones de urgencia y olas de calor — RM 65+

Análisis de atenciones de urgencia en la Región Metropolitana durante eventos de calor extremo.
Complementa el análisis de exceso de mortalidad del notebook 6.

**Fuentes:**
- Atenciones de Urgencia DEIS: 2018, 2019, 2022, 2023, 2024, 2025
- Temperatura: ERA5 reanalysis (Open-Meteo API)

**Causas de interés:**
- Sistema circulatorio (IAM, AVE, crisis hipertensiva, arritmia)
- Sistema respiratorio
- Total urgencias

---

## 0. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from scipy import stats as sp
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

DATA_DIR   = '../data/egresos/'
OUT_DIR    = '../outputs/'
FECHA_FIN  = '2025-12-31'

ANIOS_URG = [2018, 2019, 2022, 2023, 2024, 2025]

MESES_VERANO = [11, 12, 1, 2, 3]

# Causas relevantes (IDs del dataset DEIS urgencias)
# 1  = Total atenciones
# 2  = Total respiratorio
# 12 = Total circulatorio
# 13 = IAM
# 14 = AVE
# 15 = Crisis hipertensiva
# 16 = Arritmia grave
# 29 = Diarrea aguda (control — no deberia subir con calor)
CAUSAS_INTERES = [1, 2, 12, 13, 14, 15, 16, 29]

print('Configuracion lista')

## 1. Carga y estructura de datos

In [ ]:
def cargar_urgencias(anio):
    path = f'{DATA_DIR}AtencionesUrgencia{anio}.csv'
    df = pd.read_csv(path, encoding='latin-1', sep=';', low_memory=False)
    df['fecha'] = pd.to_datetime(df['fecha'], dayfirst=True, errors='coerce')
    df['anio'] = df['fecha'].dt.year
    return df

print('Cargando archivos de urgencias...')
partes = []
for anio in ANIOS_URG:
    df_a = cargar_urgencias(anio)
    partes.append(df_a)
    print(f'  {anio}: {len(df_a):,} filas | {df_a["fecha"].min().date()} - {df_a["fecha"].max().date()}')

df_all = pd.concat(partes, ignore_index=True)
print(f'\nTotal concatenado: {len(df_all):,} filas')

# Identificar establecimientos RM
# El código de establecimiento comienza con '13' para RM
df_all['IdEst_str'] = df_all['IdEstablecimiento'].astype(str)
mask_rm = df_all['IdEst_str'].str.startswith('13')
df_rm = df_all[mask_rm].copy()
print(f'Filas RM: {len(df_rm):,} ({mask_rm.mean()*100:.1f}%)')
print(f'Establecimientos RM: {df_rm["IdEstablecimiento"].nunique()}')

# Causas disponibles
print('\n=== CAUSAS DISPONIBLES ===')
causas = (df_rm[['IdCausa','GlosaCausa']]
          .drop_duplicates()
          .sort_values('IdCausa')
          .assign(GlosaCausa=lambda d: d['GlosaCausa']
                  .str.encode('latin-1', errors='replace')
                  .str.decode('utf-8', errors='replace')))
print(causas.to_string(index=False))

## 2. Serie diaria RM 65+ por causa

In [ ]:
# Agregar por fecha + causa para RM, columna 65+
df_causa = (df_rm[df_rm['IdCausa'].isin(CAUSAS_INTERES)]
            .groupby(['fecha', 'IdCausa', 'GlosaCausa'])['De_65_y_mas']
            .sum()
            .reset_index())

# Nombres cortos para graficar
NOMBRES_CAUSA = {
    1:  'Total urgencias',
    2:  'Respiratorio total',
    12: 'Circulatorio total',
    13: 'IAM',
    14: 'AVE',
    15: 'Crisis hipertensiva',
    16: 'Arritmia grave',
    29: 'Diarrea aguda (control)',
}
df_causa['nombre'] = df_causa['IdCausa'].map(NOMBRES_CAUSA)

# Pivotar a wide: una columna por causa
df_wide = df_causa.pivot_table(index='fecha', columns='nombre',
                                values='De_65_y_mas', aggfunc='sum')
df_wide.index = pd.to_datetime(df_wide.index)
df_wide['mes'] = df_wide.index.month
df_wide['anio'] = df_wide.index.year

print('Serie diaria 65+ RM por causa:')
print(df_wide.describe().round(1))

## 3. Temperatura ERA5

In [ ]:
print('Descargando ERA5...')
r = requests.get(
    'https://archive-api.open-meteo.com/v1/archive'
    '?latitude=-33.49&longitude=-70.58'
    f'&start_date=2018-01-01&end_date={FECHA_FIN}'
    '&daily=temperature_2m_max,temperature_2m_min'
    '&timezone=America%2FSantiago',
    timeout=60
)
d = r.json()
df_t = pd.DataFrame({
    'fecha': pd.to_datetime(d['daily']['time']),
    'tmax':  d['daily']['temperature_2m_max'],
    'tmin':  d['daily']['temperature_2m_min'],
}).set_index('fecha')

mask_ver = df_t.index.month.isin(MESES_VERANO)
p90 = df_t.loc[mask_ver, 'tmax'].quantile(0.90)
print(f'ERA5: {df_t.index.min().date()} - {df_t.index.max().date()}')
print(f'P90 T_max verano (2018-2025): {p90:.1f} C')

# Unir temperatura a la serie de urgencias
df_wide = df_wide.join(df_t[['tmax', 'tmin']], how='left')

## 4. Correlación T_max vs urgencias — estructura de rezago

In [ ]:
# Solo meses de verano para evitar confusión estacional (gripe invernal sube respiratorio)
df_ver = df_wide[df_wide['mes'].isin(MESES_VERANO)].copy()

causas_plot = [
    ('Circulatorio total',    '#c0392b'),
    ('IAM',                   '#e74c3c'),
    ('AVE',                   '#e67e22'),
    ('Crisis hipertensiva',   '#8e44ad'),
    ('Respiratorio total',    '#2980b9'),
    ('Diarrea aguda (control)', '#7f8c8d'),
]

lags = range(0, 8)
lag_data = {}
for causa, _ in causas_plot:
    if causa not in df_ver.columns:
        continue
    res = []
    for lag in lags:
        t_lag = df_ver['tmax'].shift(lag)
        valid = df_ver[causa].notna() & t_lag.notna()
        r, p = sp.pearsonr(t_lag[valid], df_ver[causa][valid])
        res.append({'lag': lag, 'r': r, 'p': p, 'n': valid.sum()})
    lag_data[causa] = pd.DataFrame(res)

fig, ax = plt.subplots(figsize=(10, 5))
for (causa, color) in causas_plot:
    if causa not in lag_data:
        continue
    df_lg = lag_data[causa]
    ls = '--' if causa == 'Diarrea aguda (control)' else '-'
    ax.plot(df_lg['lag'], df_lg['r'], marker='o', color=color,
            linewidth=1.8, linestyle=ls, label=causa)
    for _, row in df_lg.iterrows():
        if row['p'] < 0.05:
            ax.plot(row['lag'], row['r'], 'o', markersize=9,
                    color=color, markeredgecolor='black', markeredgewidth=1.1)

ax.axhline(0, color='grey', linewidth=0.8)
ax.set_xlabel('Rezago (dias)')
ax.set_ylabel('r de Pearson')
ax.set_title('Correlacion T$_{max}$ vs atenciones de urgencia RM 65+\n'
             'Veranos 2018-19, 2021-22, 2023-24, 2024-25 (circulos = p<0.05)')
ax.set_xticks(list(lags))
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}urg_lag_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nResumen r (lag 0):')
for causa, _ in causas_plot:
    if causa in lag_data:
        row = lag_data[causa].iloc[0]
        print(f'  {causa:<30} r={row["r"]:+.3f}  p={row["p"]:.4f}')

## 5. Exceso de urgencias durante olas de calor — event study

In [ ]:
# Olas con datos disponibles
OLAS = {
    'Ene 2019': {
        'ola_ini':  '2019-01-26', 'ola_fin':  '2019-02-03',
        'base_ini': '2019-01-05', 'base_fin': '2019-01-25',
    },
    'Dic 2022': {
        'ola_ini':  '2022-12-06', 'ola_fin':  '2022-12-13',
        'base_ini': '2022-11-10', 'base_fin': '2022-11-30',
    },
    'Feb 2023': {
        'ola_ini':  '2023-02-01', 'ola_fin':  '2023-02-15',
        'base_ini': '2023-01-10', 'base_fin': '2023-01-31',
    },
    'Ene 2024': {
        'ola_ini':  '2024-01-18', 'ola_fin':  '2024-02-10',
        'base_ini': '2023-12-20', 'base_fin': '2024-01-17',
    },
}

causas_ev = [
    'Total urgencias',
    'Circulatorio total',
    'IAM',
    'AVE',
    'Crisis hipertensiva',
    'Respiratorio total',
    'Diarrea aguda (control)',
]

resultados = []
for ola_nombre, cfg in OLAS.items():
    n_ola  = (pd.to_datetime(cfg['ola_fin'])  - pd.to_datetime(cfg['ola_ini'])).days + 1
    n_base = (pd.to_datetime(cfg['base_fin']) - pd.to_datetime(cfg['base_ini'])).days + 1
    for causa in causas_ev:
        if causa not in df_wide.columns:
            continue
        v_ola  = df_wide.loc[cfg['ola_ini']:cfg['ola_fin'],  causa]
        v_base = df_wide.loc[cfg['base_ini']:cfg['base_fin'], causa]
        rate_ola  = v_ola.sum()  / n_ola
        rate_base = v_base.sum() / n_base
        exceso_pct = (rate_ola - rate_base) / rate_base * 100 if rate_base > 0 else np.nan
        resultados.append({
            'ola': ola_nombre, 'causa': causa,
            'rate_ola': rate_ola, 'rate_base': rate_base,
            'exceso_pct': exceso_pct,
        })

df_ev = pd.DataFrame(resultados)

# Grafico tipo heatmap: causas x olas
pivot = df_ev.pivot(index='causa', columns='ola', values='exceso_pct')
pivot = pivot.reindex(index=causas_ev, columns=list(OLAS.keys()))

fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=-30, vmax=50)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, fontsize=11)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=10)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:+.0f}%', ha='center', va='center',
                    fontsize=9, fontweight='bold',
                    color='white' if abs(val) > 30 else 'black')
plt.colorbar(im, ax=ax, label='Exceso (% vs baseline)')
ax.set_title('Exceso de urgencias RM 65+ durante olas de calor\n'
             '(% vs periodo baseline equivalente, causa x evento)')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}urg_exceso_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print(pivot.round(1).to_string())

## 6. Series temporales alrededor de las olas

In [ ]:
# Ventana de +-45 dias alrededor de cada ola para cada causa principal
VENTANA = 45  # dias

causas_ts = ['Circulatorio total', 'IAM', 'AVE', 'Respiratorio total']
colores_ts = ['#c0392b', '#e74c3c', '#e67e22', '#2980b9']

olas_ts = [
    ('Ene 2019', pd.Timestamp('2019-01-26'), pd.Timestamp('2019-02-03')),
    ('Dic 2022', pd.Timestamp('2022-12-06'), pd.Timestamp('2022-12-13')),
    ('Feb 2023', pd.Timestamp('2023-02-01'), pd.Timestamp('2023-02-15')),
    ('Ene 2024', pd.Timestamp('2024-01-18'), pd.Timestamp('2024-02-10')),
]

fig, axes = plt.subplots(len(olas_ts), len(causas_ts),
                          figsize=(16, 13), sharex='row')

for row_i, (ola_nom, ola_ini, ola_fin) in enumerate(olas_ts):
    ini_v = ola_ini - pd.Timedelta(days=VENTANA)
    fin_v = ola_fin + pd.Timedelta(days=VENTANA)
    serie_v = df_wide.loc[ini_v:fin_v]

    for col_i, (causa, color) in enumerate(zip(causas_ts, colores_ts)):
        ax = axes[row_i][col_i]
        if causa not in serie_v.columns:
            ax.set_visible(False)
            continue
        ax.plot(serie_v.index, serie_v[causa],
                color=color, linewidth=1.2, alpha=0.8)
        ax.plot(serie_v.index, serie_v[causa].rolling(7, center=True).mean(),
                color=color, linewidth=2.2)
        ax.axvspan(ola_ini, ola_fin, color='orange', alpha=0.25)
        ax.axhline(serie_v.loc[ini_v:ola_ini, causa].mean(),
                   color='grey', linewidth=0.9, linestyle='--')
        if row_i == 0:
            ax.set_title(causa, fontsize=10, fontweight='bold')
        if col_i == 0:
            ax.set_ylabel(ola_nom, fontsize=10, fontweight='bold')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=3))
        plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=7)

fig.suptitle('Atenciones de urgencia RM 65+ — ventana +-45 dias por ola\n'
             '(linea delgada = diario, gruesa = media movil 7d, naranja = ola, gris = baseline pre-ola)',
             fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}urg_series_olas.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Nota metodológica

> **Años COVID (2020–2021):** excluidos del análisis porque la pandemia distorsionó
> los patrones de búsqueda de atención (subregistro de causas no-COVID y colapso de la
> atención de urgencia no-respiratoria). Incluirlos contaminaría tanto la correlación
> T_max → urgencias como los baselines de comparación por ola.